# Approach
Compared two pipelines:
1. Word2Vec (trained from scratch) + mean-pooled embeddings → Dense Neural Network
2. TF-IDF (unigrams + bigrams) → Logistic Regression (`class_weight='balanced'`)

## Results

| Model | Accuracy | Macro F1 |
|---|---|---|
| Word2Vec + Dense NN | 68.7% | 0.56 |
| TF-IDF + Logistic Regression | 79.0% | 0.73 

In [2]:
import pandas as pd
import tensorflow as tf
import numpy as np

In [3]:
data = pd.read_csv("/mnt/c/Users/User/Downloads/airline_tweets.csv")
data

,tweet_id,airline_sentiment,airline_sentiment_confidence,negativereason,negativereason_confidence,airline,airline_sentiment_gold,name,negativereason_gold,retweet_count,text,tweet_coord,tweet_created,tweet_location,user_timezone
0,570306133677760513,neutral,1.0000,NaN,NaN,Virgin America,NaN,cairdin,NaN,0,@VirginAmerica What @dhepburn said.,NaN,2015-02-24 11:35:52 -0800,NaN,Eastern Time (US & Canada)
1,570301130888122368,positive,0.3486,NaN,0.0000,Virgin America,NaN,jnardino,NaN,0,@VirginAmerica plus you've added commercials t...,NaN,2015-02-24 11:15:59 -0800,NaN,Pacific Time (US & Canada)
2,570301083672813571,neutral,0.6837,NaN,NaN,Virgin America,NaN,yvonnalynn,NaN,0,@VirginAmerica I didn't today... Must mean I n...,NaN,2015-02-24 11:15:48 -0800,Lets Play,Central Time (US & Canada)
3,570301031407624196,negative,1.0000,Bad Flight,0.7033,Virgin America,NaN,jnardino,NaN,0,@VirginAmerica it's really aggressive to blast...,NaN,2015-02-24 11:15:36 -0800,NaN,Pacific Time (US & Canada)
4,570300817074462722,negative,1.0000,Can't Tell,1.0000,Virgin America,NaN,jnardino,NaN,0,@VirginAmerica and it's a really big bad thing...,NaN,2015-02-24 11:14:45 -0800,NaN,Pacific Time (US & Canada)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14635,569587686496825344,positive,0.3487,NaN,0.0000,American,NaN,KristenReenders,NaN,0,@AmericanAir thank you we got on a different f...,NaN,2015-02-22 12:01:01 -0800,NaN,NaN
14636,569587371693355008,negative,1.0000,Customer Service Issue,1.0000,American,NaN,itsropes,NaN,0,@AmericanAir leaving over 20 minutes Late Flig...,NaN,2015-02-22 11:59:46 -0800,Texas,NaN
14637,569587242672398336,neutral,1.0000,NaN,NaN,American,NaN,sanyabun,NaN,0,@AmericanAir Please bring American Airlines to...,NaN,2015-02-22 11:59:15 -0800,"Nigeria,lagos",NaN
14638,569587188687634433,negative,1.0000,Customer Service Issue,0.6659,American,NaN,SraJackson,NaN,0,"@AmericanAir you have my money, you change my ...",NaN,2015-02-22 11:59:02 -0800,New Jersey,Eastern Time (US & Canada)


In [4]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 14640 entries, 0 to 14639
Data columns (total 15 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   tweet_id                      14640 non-null  int64  
 1   airline_sentiment             14640 non-null  str    
 2   airline_sentiment_confidence  14640 non-null  float64
 3   negativereason                9178 non-null   str    
 4   negativereason_confidence     10522 non-null  float64
 5   airline                       14640 non-null  str    
 6   airline_sentiment_gold        40 non-null     str    
 7   name                          14640 non-null  str    
 8   negativereason_gold           32 non-null     str    
 9   retweet_count                 14640 non-null  int64  
 10  text                          14640 non-null  str    
 11  tweet_coord                   1019 non-null   str    
 12  tweet_created                 14640 non-null  str    
 13  tweet_locati

In [5]:
data.isna().sum()

tweet_id                            0
airline_sentiment                   0
airline_sentiment_confidence        0
negativereason                   5462
negativereason_confidence        4118
airline                             0
airline_sentiment_gold          14600
name                                0
negativereason_gold             14608
retweet_count                       0
text                                0
tweet_coord                     13621
tweet_created                       0
tweet_location                   4733
user_timezone                    4820
dtype: int64

In [6]:
data.drop(['airline_sentiment_gold', 'negativereason_gold','tweet_coord','tweet_id','name'], axis=1, inplace = True)

In [8]:
import nltk
import spacy
import gensim
from sklearn.feature_extraction.text import TfidfVectorizer

In [9]:
data.drop(['tweet_created','tweet_location','user_timezone'], axis=1, inplace = True)
data

,airline_sentiment,airline_sentiment_confidence,negativereason,negativereason_confidence,airline,retweet_count,text
0,neutral,1.0000,NaN,NaN,Virgin America,0,@VirginAmerica What @dhepburn said.
1,positive,0.3486,NaN,0.0000,Virgin America,0,@VirginAmerica plus you've added commercials t...
2,neutral,0.6837,NaN,NaN,Virgin America,0,@VirginAmerica I didn't today... Must mean I n...
3,negative,1.0000,Bad Flight,0.7033,Virgin America,0,@VirginAmerica it's really aggressive to blast...
4,negative,1.0000,Can't Tell,1.0000,Virgin America,0,@VirginAmerica and it's a really big bad thing...
...,...,...,...,...,...,...,...
14635,positive,0.3487,NaN,0.0000,American,0,@AmericanAir thank you we got on a different f...
14636,negative,1.0000,Customer Service Issue,1.0000,American,0,@AmericanAir leaving over 20 minutes Late Flig...
14637,neutral,1.0000,NaN,NaN,American,0,@AmericanAir Please bring American Airlines to...
14638,negative,1.0000,Customer Service Issue,0.6659,American,0,"@AmericanAir you have my money, you change my ..."


In [10]:
data.text.isna().sum()

np.int64(0)

In [11]:
from nltk.tokenize import word_tokenize
from gensim.models import FastText, Word2Vec

In [12]:
def tokenize_(text):
    tokens = word_tokenize(text.lower())
    return [t for t in tokens if t.isalpha()]

In [13]:
x = data.text
y = data.airline_sentiment

In [14]:
tokenized_data = [tokenize_(s) for s in x]

In [15]:
word_tokenize('i hate you')

['i', 'hate', 'you']

In [16]:
tokenized_data[:10]

[['virginamerica', 'what', 'dhepburn', 'said'],
 ['virginamerica',
  'plus',
  'you',
  'added',
  'commercials',
  'to',
  'the',
  'experience',
  'tacky'],
 ['virginamerica',
  'i',
  'did',
  'today',
  'must',
  'mean',
  'i',
  'need',
  'to',
  'take',
  'another',
  'trip'],
 ['virginamerica',
  'it',
  'really',
  'aggressive',
  'to',
  'blast',
  'obnoxious',
  'entertainment',
  'in',
  'your',
  'guests',
  'faces',
  'amp',
  'they',
  'have',
  'little',
  'recourse'],
 ['virginamerica',
  'and',
  'it',
  'a',
  'really',
  'big',
  'bad',
  'thing',
  'about',
  'it'],
 ['virginamerica',
  'seriously',
  'would',
  'pay',
  'a',
  'flight',
  'for',
  'seats',
  'that',
  'did',
  'have',
  'this',
  'playing',
  'it',
  'really',
  'the',
  'only',
  'bad',
  'thing',
  'about',
  'flying',
  'va'],
 ['virginamerica',
  'yes',
  'nearly',
  'every',
  'time',
  'i',
  'fly',
  'vx',
  'this',
  'ear',
  'worm',
  'won',
  't',
  'go',
  'away'],
 ['virginamerica',
  '

In [17]:
model_w2v = Word2Vec(sentences=tokenized_data, vector_size = 100, window = 5, min_count=1, workers = 4)

In [19]:
def get_sentence_embedding(tokens, model):
    valid_vectors = [model.wv[word] for word in tokens if word in model.wv]
    if not valid_vectors:
        return np.zeros(model.vector_size)
    return np.mean(valid_vectors, axis = 0)

In [20]:
model_w2v.wv['hello']

array([-0.08582744,  0.22417958,  0.05485775, -0.04459953, -0.0264007 ,
       -0.2743391 ,  0.09197191,  0.38856396, -0.09016099, -0.06410222,
       -0.06825648, -0.20687728, -0.08630817,  0.04417361,  0.08566172,
       -0.06020165,  0.03989608, -0.11258266, -0.08875119, -0.28721783,
       -0.00400339,  0.02143761,  0.17940788, -0.01113488, -0.00982443,
       -0.01885611, -0.10360069, -0.16322853, -0.09193737,  0.0523213 ,
        0.16431527, -0.01826488, -0.0085757 , -0.20989755, -0.03886098,
        0.15136571,  0.13017832, -0.04836803, -0.01219087, -0.32858673,
        0.07911324, -0.14666754, -0.07846364, -0.01952452,  0.15371002,
       -0.0495022 , -0.22951663,  0.07403754,  0.19961001,  0.22382285,
        0.07540113, -0.17126837,  0.03415477, -0.09373518, -0.11070249,
        0.17527832,  0.21019836, -0.05402796, -0.14330786, -0.03299017,
       -0.01521832,  0.160694  ,  0.11171649, -0.07631492, -0.24126078,
        0.22370425,  0.05488144,  0.25485623, -0.21418941,  0.27

In [21]:
embeddings = np.array([get_sentence_embedding(s, model_w2v) for s in tokenized_data])

In [22]:
embeddings.shape

(14640, 100)

In [23]:
from tensorflow.keras.layers import Input, Dense

In [24]:
data.airline_sentiment.value_counts()

airline_sentiment
negative    9178
neutral     3099
positive    2363
Name: count, dtype: int64

In [25]:
model = tf.keras.Sequential([
    Input(shape=(100,)),
    Dense(128, activation="relu"),
    Dense(64, activation="relu"),
    Dense(3, activation="softmax")
])

E0000 00:00:1781775924.657268   18782 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [26]:
classes = {
    'negative': 0,
    'neutral': 1,
    'positive': 2
}

y = y.map(classes)

In [27]:
model.compile(optimizer='adam', loss = 'sparse_categorical_crossentropy', metrics = ['accuracy'])

In [28]:
embeddings

array([[-0.1873461 ,  0.39436996,  0.07943337, ..., -0.38985476,
         0.05599947, -0.03860024],
       [-0.22858378,  0.5567898 ,  0.05454532, ..., -0.34637544,
         0.09477346,  0.03199939],
       [-0.3203208 ,  0.8083671 ,  0.10395822, ..., -0.43492815,
         0.2489894 ,  0.00149919],
       ...,
       [-0.26571578,  0.71745014,  0.11803766, ..., -0.44892284,
         0.15542343, -0.09272069],
       [-0.2918168 ,  0.7668767 ,  0.07701232, ..., -0.40960196,
         0.27565908, -0.06926595],
       [-0.24736717,  0.8066131 ,  0.05985952, ..., -0.3173607 ,
         0.24212442,  0.01419009]], shape=(14640, 100), dtype=float32)

In [29]:
from sklearn.model_selection import train_test_split

In [30]:
x_train, x_test, y_train, y_test = train_test_split(embeddings, y, test_size = 0.2, random_state=0)

In [31]:
model.fit(x_train, y_train, epochs = 3, batch_size = 32, validation_data=(x_test,y_test))

Epoch 1/3
366/366 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.6552 - loss: 0.8180 - val_accuracy: 0.6742 - val_loss: 0.7657
Epoch 2/3
366/366 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.6814 - loss: 0.7615 - val_accuracy: 0.6885 - val_loss: 0.7470
Epoch 3/3
366/366 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.6926 - loss: 0.7485 - val_accuracy: 0.6868 - val_loss: 0.7541


In [32]:
labels = {0:'negative',1:'neutral',2:'positive'}

In [34]:
labels[np.argmax(model.predict(get_sentence_embedding(word_tokenize('i hate this airline so much '), model_w2v).reshape(1,-1)))]

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step


'positive'

In [35]:
data.airline.value_counts()

airline
United            3822
US Airways        2913
American          2759
Southwest         2420
Delta             2222
Virgin America     504
Name: count, dtype: int64

In [38]:
from sklearn.metrics import classification_report
print(classification_report(y_test, model.predict(x_test).argmax(axis=1)))

92/92 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
              precision    recall  f1-score   support

           0       0.75      0.86      0.80      1870
           1       0.51      0.35      0.42       614
           2       0.52      0.41      0.46       444

    accuracy                           0.69      2928
   macro avg       0.59      0.54      0.56      2928
weighted avg       0.66      0.69      0.67      2928



In [40]:
x,y

(0                      @VirginAmerica What @dhepburn said.
 1        @VirginAmerica plus you've added commercials t...
 2        @VirginAmerica I didn't today... Must mean I n...
 3        @VirginAmerica it's really aggressive to blast...
 4        @VirginAmerica and it's a really big bad thing...
                                ...                        
 14635    @AmericanAir thank you we got on a different f...
 14636    @AmericanAir leaving over 20 minutes Late Flig...
 14637    @AmericanAir Please bring American Airlines to...
 14638    @AmericanAir you have my money, you change my ...
 14639    @AmericanAir we have 8 ppl so we need 2 know h...
 Name: text, Length: 14640, dtype: str,
 0        1
 1        2
 2        1
 3        0
 4        0
         ..
 14635    2
 14636    0
 14637    1
 14638    0
 14639    1
 Name: airline_sentiment, Length: 14640, dtype: int64)

In [64]:
x_train_, x_test_, y_train_, y_test_ = train_test_split(x, y, stratify=y, test_size = 0.2, random_state=0)

In [65]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

In [66]:
tfidf = TfidfVectorizer(lowercase = True, ngram_range = (1,2))
x_train_vec = tfidf.fit_transform(x_train_)
x_test_vec = tfidf.transform(x_test_)

In [100]:
l = LogisticRegression(class_weight='balanced')
l.fit(x_train_vec,y_train_)
print(classification_report(y_test_, l.predict(x_test_vec)))

              precision    recall  f1-score   support

           0       0.87      0.87      0.87      1835
           1       0.61      0.63      0.62       620
           2       0.71      0.70      0.71       473

    accuracy                           0.79      2928
   macro avg       0.73      0.73      0.73      2928
weighted avg       0.79      0.79      0.79      2928



In [70]:
labels[l.predict(tfidf.transform(['i hate this']))[0]]

'negative'

In [71]:
labels

{0: 'negative', 1: 'neutral', 2: 'positive'}

In [97]:
ypr = l.predict(x_test_vec)
mask1 = ((y_test_.values==1) & (ypr!=1))
x_test_orig = x_test_[mask1]
labell = y_test_[mask1]
yprr = ypr[mask1]
for i in range(10):
    print('Original Sentence')
    print(x_test_orig.iloc[i])
    print(f'Real - {labels[labell.iloc[i]]}, Pred - {labels[yprr[i]]}')
    print(50*'_')

Original Sentence
@AmericanAir #AmericanAirlines on approach to mex.... http://t.co/se57dmjHiW
Real - neutral, Pred - negative
__________________________________________________
Original Sentence
@united, Understandable. I did try Flight Booking Problems several times for 2 passengers &amp; got the messages I mentioned before. As for the agents price???
Real - neutral, Pred - negative
__________________________________________________
Original Sentence
@JetBlue  Not blaming Jet Blue.  This wasn't weather. Can't have planes in the air and runways a mess. That's a disaster waiting to happen.
Real - neutral, Pred - negative
__________________________________________________
Original Sentence
@united hey, I missed my outbound flight - can I still use my return ticket?
Real - neutral, Pred - negative
__________________________________________________
Original Sentence
@JetBlue I sure hope you guys get me to DC to speak tomorrow! &amp; @JohnNosta &amp; @United: I'm winning here with @JetBlue

In [98]:
mask1 = ((y_test_.values==0) & (ypr!=0))
x_test_orig = x_test_[mask1]
labell = y_test_[mask1]
yprr = ypr[mask1]
for i in range(10):
    print('Original Sentence')
    print(x_test_orig.iloc[i])
    print(f'Real - {labels[labell.iloc[i]]}, Pred - {labels[yprr[i]]}')
    print(50*'_')

Original Sentence
@JetBlue but the 4 hour policy- when I called and they said I didn't report it soon enough. What would have changed if I had noticed sooner?
Real - negative, Pred - positive
__________________________________________________
Original Sentence
@USAirways Houston hub Aa T. Employee the bag is not here, It made it to Pa before me! Oops not your problem http://t.co/Uld0hWFkfo
Real - negative, Pred - positive
__________________________________________________
Original Sentence
@united free hotel doesn't make up for such poor customer service and disregard
Real - negative, Pred - positive
__________________________________________________
Original Sentence
@SouthwestAir sitting in Baltimore. Finish taxi and approved to takeoff. Oops, not enough fuel. #Fail #45minflight #been3hours
Real - negative, Pred - neutral
__________________________________________________
Original Sentence
@AmericanAir sitting on plane in Columbus, supposed to leave an hour ago. Now the mechanic can'

In [99]:
mask1 = ((y_test_.values==2) & (ypr!=2))
x_test_orig = x_test_[mask1]
labell = y_test_[mask1]
yprr = ypr[mask1]
for i in range(10):
    print('Original Sentence')
    print(x_test_orig.iloc[i])
    print(f'Real - {labels[labell.iloc[i]]}, Pred - {labels[yprr[i]]}')
    print(50*'_')

Original Sentence
@AmericanAir fantastic thanks! Will try and tweet a photo of the view :)
Real - positive, Pred - negative
__________________________________________________
Original Sentence
@SouthwestAir JH thank you. I finally got through the second time.
Real - positive, Pred - negative
__________________________________________________
Original Sentence
@SouthwestAir props to your LAS employees working C11 gate. Because of them I am not opposed to flying through or to LAS in the future! 👏👏👏
Real - positive, Pred - negative
__________________________________________________
Original Sentence
@USAirways your ticket agents at gate 4 in Providence airport rocked tonight, especially Kristy, sorry if that is not the correct spelling.
Real - positive, Pred - neutral
__________________________________________________
Original Sentence
@VirginAmerica got it squared away. Someone picked up as soon as I tweeted. Should have tweeted sooner. 😉
Real - positive, Pred - negative
________________